In [32]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

# 1. Seleccion del dataset
Es importante saber que el repositorio del data set nos dice que ninguna de las columnas contienen valores faltantes, lo que nos facilita la tarea que vamos a llevar a cabo

In [ ]:
wine_raw = load_wine()
x = wine_raw.data
y = wine_raw.target

print(set(y))

Sabemos que los vinos se clasifican en 3 posibles categorias, mapeemoslos dentro del dataset

In [ ]:
df = pd.DataFrame(x, columns=wine_raw.feature_names)
df['target'] = y
df['cultivar'] = 'Cultivar ' + df['target'].astype(str)

print(f'Dimensiones del dataset: {df.shape}')
print(f'\nFilas duplicadas: {df.duplicated().sum()}')
print('\nEstadísticas descriptivas:\n', df.describe().T[['mean', 'std', 'min', 'max']])

df.head()

# 2. Comprension del dataset

Según la informacion del repositorio del data set:

El dataset contiene 178 muestras de vinos, 13 variables fisicoquímicas y una variable objetivo con 3 clases.

Las 13 variables son:
| Variable | ¿Qué representa? |
| --- | --- |
|alcohol | Contenido de alcohol. |
malic_acid	|Concentración de ácido málico.
ash	|Cantidad de cenizas.
alcalinity_of_ash	|Alcalinidad de las cenizas.
magnesium	|Concentración de magnesio.
total_phenols	|Concentración total de fenoles.
flavanoids	|Concentración de flavonoides.
nonflavanoid_phenols	|Concentración de fenoles no flavonoides.
proanthocyanins	|Concentración de proantocianidinas.
color_intensity	|Intensidad del color.
hue	|Matiz o tonalidad del color.
od280/od315_of_diluted_wines|	Relación de absorbancia óptica.
proline|Concentración de prolina.

La variable objetivo es target, que identifica el cultivar al que pertenece cada vino, este target y la clase que representa son las que hemos añadido al dataset utilizando el siguiente mapeo:
| target | Cultivar |
| --- | --- |
0 | Cultivar 0
1 | Cultivar 1
2 | Cultivar 2

Realicemos un análisis grafico de las distintas muestras por poblacion que tenemos

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x='cultivar',
    hue='cultivar',
    palette='viridis',
    legend=False
)

plt.title('Distribución de poblaciones')
plt.xlabel('Cultivar')
plt.ylabel('Cantidad de muestras')
plt.show()

Hay una gran diferencia entre la poblacion con el mayor número de muestras (cultivar 1) y la poblacion con el menor número de muestras (cultivar 2).
Ahora veamos si la correlacion entre las distintas variables fisicoquimicas, no se trata de poblaciones balanceadas lo que puede dificultar un chin la capacidad de clasificacion, ojo.

Veamos si para el caso de las caracteristicas fisicoquimicas se presentan algun tipo de asimetrias o outliners

In [ ]:
X = df.drop(columns=['target', 'cultivar'])

X.hist(
    figsize=(16, 12),
    bins=20,
    edgecolor='black'
)

plt.suptitle('Distribución de las variables fisicoquímicas')
plt.tight_layout()
plt.show()

Es curioso analizar que a excepción de nonflavanoid_phenols que presenta un hueco entre 0.3 y 0.4, todos se distribuyen aproximadamente de forma normal

Analicemos la posible correlacion entre todas las distintas variables fisicoquimicas del dataset

In [ ]:
plt.figure(figsize=(13, 10))

sns.heatmap(
    X.corr(),
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title('Correlación entre variables fisicoquímicas')
plt.tight_layout()
plt.show()

Ahora la relacion entre variables y poblaciones utilizando un pairplot, espero q les sirva niños

In [ ]:
sns.pairplot(
    df,
    vars=[
        'alcohol',
        'flavanoids',
        'color_intensity',
        'proline'
    ],
    hue='cultivar',
    palette='viridis',
    diag_kind='hist'
)

plt.show()

Como podemos ver en los graficos aunque en muchos casos hay bastantes superpuestos siguen pudiendose distinguir de manera muy clara como se agrupan según la poblacion a la que pertenece la muestra, veamos que caracteristicas definen más a una poblacion que a otra

In [ ]:
# Chao no predictorias como siempre
features = df.drop(columns=['target', 'cultivar']).columns

fig, axes = plt.subplots(4, 4, figsize=(18, 15))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        data=df,
        x='cultivar',
        y=feature,
        hue='cultivar',
        palette='viridis',
        legend=False,
        ax=axes[i]
    )

    axes[i].set_title(feature)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Eliminamos los espacios sobrantes
for i in range(len(features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle(
    'Distribución de características fisicoquímicas por cultivar',
    fontsize=16
)

plt.tight_layout()
plt.show()

# 3. Limpieza y adecuacion de datos

### 3.1) Seleccion de variables predictorias
Como hemos visto en el analisis anterior de los boxplot hay ciertas variables que permiten diferenciar de manera bastante clara si una muestra pertenece o no a una poblacion, dentro de estas variables con mayor separacion entre cultivares encontramos:
- flavanoids: Util para distinguir los 3 cultivares
- hue: Tiene el cultivar 2 muy marcado
- od280/od315_of_diluted_wines: Tambien tiene marcada la diferencia del cultivar 2 muy marcada
- proline: Destaca la diferencia del cultivar 0
- color_intensity: Muestra diferentes cheveres entre los 3 grupos
- malic_acid: Segmenta el cultivar 2
- total_phenols: Muestra una buena diferencia de los 3 cultivares
- alcohol: Segmenta muy bien el cultivar 0 del 1

Dado que hasta el momento tenemos claras cuales vamos a usar pero no cuales podemos descartar hagamos un analisis de varianza para determinar si hay diferencias estadisticas significativas, analizaremos estas 2 variables:
- F-statistic: cuanto mayor sea, mayor es la variabilidad entre las medias de los grupos en relación con la variabilidad dentro de ellos.
- p-value: si es inferior a 0.05, encontramos evidencia estadística de que no todas las medias son iguales.

In [ ]:
from scipy.stats import f_oneway

features = df.drop(columns=['target', 'cultivar']).columns

resultados = []

for feature in features:
    grupos = [
        df.loc[df['target'] == clase, feature]
        for clase in sorted(df['target'].unique())
    ]

    f_statistic, p_value = f_oneway(*grupos)

    resultados.append({
        'Variable': feature,
        'F-statistic': f_statistic,
        'p-value': p_value,
        'Significativa': p_value < 0.05
    })

anova_df = pd.DataFrame(resultados)

anova_df = anova_df.sort_values(
    by='F-statistic',
    ascending=False
)

print(anova_df.round(6).to_string(index=False))

Para este caso obtenemos que variables como ash o magnesium tienen una muy pobre capacidad discriminativa si la comparamos con flavanoids por ejemplo, así que para este caso hemos decidido dividir el conjunto de datos en 2 grupos según su capacidad discriminativa:
- Mayor capacidad discriminativa: flavanoids, proline, od280/od315_of_diluted_wines, alcohol, color_intensity, hue y total_phenols.
- Menor capacidad discriminativa individual: malic_acid, alcalinity_of_ash, proanthocyanins, nonflavanoid_phenols, ash y magnesium.

Lo realizamos de esta manera porque a pesar de que haya una variable con poca capacidad discriminativa inicial puede aportar al combinarse con otras.


In [ ]:
variables_fuertes = [
    'flavanoids',
    'proline',
    'od280/od315_of_diluted_wines',
    'alcohol',
    'color_intensity',
    'hue',
    'total_phenols'
]
variables_debiles = [
    'malic_acid',
    'alcalinity_of_ash',
    'proanthocyanins',
    'nonflavanoid_phenols',
    'ash',
    'magnesium'
]
# Variable objetivo
y = df['target']

# Dataset completo
X_completo = df.drop(columns=['target', 'cultivar'])

# Dataset con variables de mayor capacidad discriminativa
X_fuertes = df[variables_fuertes]

# Dataset con variables de menor capacidad discriminativa
X_debiles = df[variables_debiles]

# Verificar dimensiones
print("Dataset completo:", X_completo.shape)
print("Dataset variables fuertes:", X_fuertes.shape)
print("Dataset variables débiles:", X_debiles.shape)

### 3.2) Estandarizacin o normalizacion de las variables
Nos damos cuenta que las variables del dataset tienen escalas muy diferentes, lo que resulta un problema para KNN ya que variables aportarian más que otras, por eso decidimos estandarizar las variables con la transdformacion StandarScaler
$$
z = \frac{x - \mu}{\sigma}
$$
Para llevar las variables a una escala comun utilizando la media y desviacion estandar, despues de esta transformacion cad variable tendra aproximadamente
- Media  = 0
- Desviacion estandar = 1


# 4. Construccion del dataset
Vamos a construir y exportar el dataset a csv utilizando tamaños de 20% y 30% y aqui vamos a aplicar la estandarizacion, ya que si estandarizamos antes de dividir las estadisticas de observaciones de prueba participan en el calculo de media y desviacion estandar, mientras que si dividimos y despues partimos este aprende exclusivamente del entrenamiento

In [30]:
# Variable objetivo
y = df['target']

# Los tres datasets originales, sin estandarizar
datasets = {
    'completo': X_completo,
    'fuertes': X_fuertes,
    'debiles': X_debiles
}

# Porcentajes de prueba
porcentajes = [0.20, 0.30]

# Almacenar los resultados
particiones = {}

for porcentaje in porcentajes:

    # Dividir los índices una sola vez
    idx_train, idx_test = train_test_split(
        df.index,
        test_size=porcentaje,
        random_state=42,
        stratify=y
    )

    for nombre, X in datasets.items():

        # Separar entrenamiento y prueba
        X_train = X.loc[idx_train]
        X_test = X.loc[idx_test]

        y_train = y.loc[idx_train]
        y_test = y.loc[idx_test]

        # Estandarizar utilizando únicamente entrenamiento
        scaler = StandardScaler()

        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train),
            columns=X.columns,
            index=X_train.index
        )

        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test),
            columns=X.columns,
            index=X_test.index
        )

        # Incorporar la variable objetivo
        train = X_train_scaled.copy()
        train['target'] = y_train

        test = X_test_scaled.copy()
        test['target'] = y_test

        # Identificador de la partición
        clave = f'{nombre}_{int(porcentaje * 100)}'

        particiones[clave] = {
            'train': train,
            'test': test
        }

        print(f"\nDataset: {clave}")
        print(f"Entrenamiento: {train.shape}")
        print(f"Prueba: {test.shape}")


Dataset: completo_20
Entrenamiento: (142, 14)
Prueba: (36, 14)

Dataset: fuertes_20
Entrenamiento: (142, 8)
Prueba: (36, 8)

Dataset: debiles_20
Entrenamiento: (142, 7)
Prueba: (36, 7)

Dataset: completo_30
Entrenamiento: (124, 14)
Prueba: (54, 14)

Dataset: fuertes_30
Entrenamiento: (124, 8)
Prueba: (54, 8)

Dataset: debiles_30
Entrenamiento: (124, 7)
Prueba: (54, 7)


In [33]:
# Crear carpeta para los archivos
os.makedirs('datasets_clasificacion', exist_ok=True)

for nombre, conjuntos in particiones.items():

    conjuntos['train'].to_csv(
        f'datasets_clasificacion/{nombre}_train.csv',
        index=False
    )

    conjuntos['test'].to_csv(
        f'datasets_clasificacion/{nombre}_test.csv',
        index=False
    )

print("Todos los datasets fueron exportados correctamente.")

Todos los datasets fueron exportados correctamente.
